In [3]:
# 3.8.8

import cv2
import mediapipe as mp
import numpy as np
import random 



particles = []
clear_was_active = False



drawing = False
current_shape = []
shapes = []
selected_shape = None
dragging = False
drawing_layer = None
prev_x = None
prev_y = None
move_prev_x = None
move_prev_y = None

mp_hands=mp.solutions.hands
hands=mp_hands.Hands(
  max_num_hands=1,
  min_detection_confidence=0.5,
  min_tracking_confidence=0.5
)

mp_draw=mp.solutions.drawing_utils

hand_style=mp_draw.DrawingSpec(
  color=(105,201,40),
  circle_radius=5,
  thickness=1
)

conection_style=mp_draw.DrawingSpec(
  color=(20,150,120),
  thickness=1
)
cap=cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1040)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT,760)


def place_image(frame, image, x, y):
    h, w = image.shape[:2]
    x1 = x - w // 2
    y1 = y - h // 2
    x2 = x1 + w
    y2 = y1 + h

    if x1 < 0 or y1 < 0 or x2 > frame.shape[1] or y2 > frame.shape[0]:
        return

    if image.shape[2] == 4:
        alpha = image[:, :, 3] / 255.0
        for c in range(3):
            frame[y1:y2, x1:x2, c] = (
                alpha * image[:, :, c] +
                (1 - alpha) * frame[y1:y2, x1:x2, c]
            )
    else:
        frame[y1:y2, x1:x2] = image[:, :, :3]



def get_hand_gesture(hand_landmark):
    landmarks=hand_landmark.landmark
    index_up = landmarks[8].y < landmarks[6].y
    middle_up = landmarks[12].y < landmarks[10].y
    ring_up = landmarks[16].y < landmarks[14].y
    pinky_up = landmarks[20].y < landmarks[18].y
    thumb_up = landmarks[4].x < landmarks[2].x
    index=landmarks[8]
    thumb=landmarks[4]
    pinch_distance = np.hypot(
        thumb.x - index.x,
        thumb.y - index.y
    )
    
    if index_up and not middle_up and not thumb_up and not ring_up and not pinky_up :
        return "DRAW"
    if index_up and thumb_up :
      if pinch_distance < 0.04:
        return "MOVE"
    if index_up and  thumb_up and    middle_up and  ring_up and  pinky_up:
        return "CLEAR"
    return None
def rotate_image(image, angle):
    h, w = image.shape[:2]

    center = (w // 2, h // 2)

    matrix = cv2.getRotationMatrix2D(center, angle, 1)

    rotated = cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0, 0)
    )

    return rotated
def get_selected_shape(x, y):
    for shape in reversed(shapes):
        for sx, sy, stamp, angle, speed in shape:
            if np.hypot(x - sx, y - sy ) < 20:
                return shape
    return None

while True:


  ret,frame=cap.read()
  if not ret:
    break

  frame = cv2.flip(frame, 1)
  if drawing_layer is None:
    drawing_layer = np.zeros_like(frame)
  h,w,_=frame.shape

  frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

  results=hands.process(frame_rgb)
  if results.multi_hand_landmarks:
      for hand_landmark in results.multi_hand_landmarks:
              gesture=get_hand_gesture(hand_landmark)

              index_finger=hand_landmark.landmark[8]
              thumb_finger=hand_landmark.landmark[4]
              x=int(index_finger.x*w)
              y=int(index_finger.y*h)
              move_x = int((thumb_finger.x + index_finger.x) / 2 * w)
              move_y = int((thumb_finger.y + index_finger.y) / 2 * h)
              mp_draw.draw_landmarks(
                frame,
                hand_landmark,
                mp_hands.HAND_CONNECTIONS,
                hand_style,
                conection_style)
              if gesture == "DRAW":
                if not drawing:
                    current_shape = []
                    drawing = True
                if prev_x is None or prev_y is None:
                    randimg = random.randint(1, 24)
                    stamp = cv2.imread(f"stars/{randimg}.png", cv2.IMREAD_UNCHANGED)
                    stamp = cv2.resize(stamp, (35, 35))
                    angle = random.randint(0, 360)
                    rotation_speed = random.choice([-3, -2, -1, 1, 2, 3])
                    current_shape.append((x, y, stamp, angle, rotation_speed))


                    prev_x = x
                    prev_y = y

                else:
                    distance = np.hypot(x - prev_x, y - prev_y)

                    if distance >= 20:
                        randimg = random.randint(1, 24)
                        stamp = cv2.imread(f"stars/{randimg}.png", cv2.IMREAD_UNCHANGED)
                        stamp = cv2.resize(stamp, (35, 35))
                        angle = random.randint(0, 360)
                        rotation_speed = random.choice([-3, -2, -1, 1, 2, 3])
                        current_shape.append((x, y, stamp, angle, rotation_speed))


                        prev_x = x
                        prev_y = y
     
              
              if gesture != "DRAW" and drawing:
                if current_shape:
                    shapes.append(current_shape)
                current_shape = []
                drawing = False

              if gesture == "MOVE":
                if not dragging:
                    selected_shape = get_selected_shape(move_x, move_y)
                    if selected_shape is not None:
                        dragging = True
                        move_prev_x = move_x
                        move_prev_y = move_y
                else: 
                  if selected_shape is not None:
                    dx = move_x - move_prev_x
                    dy = move_y - move_prev_y
                    for i in range(len(selected_shape)):
                      sx, sy, stamp, angle, speed = selected_shape[i]
                      selected_shape[i] = (sx + dx, sy + dy, stamp, angle, speed)
                  move_prev_x = move_x
                  move_prev_y = move_y
              if gesture != "MOVE" and gesture!="DRAW":
                dragging = False
                selected_shape = None
                move_prev_x = None
                move_prev_y = None
              if gesture == "CLEAR" and not clear_was_active:

                  for shape in shapes:
                      for sx, sy, stamp, angle, speed in shape:
                          for i in range(12):
                              angle = random.uniform(0, 2 * np.pi)
                              speed = random.uniform(2, 7)
                              vx = np.cos(angle) * speed
                              vy = np.sin(angle) * speed
                              color = tuple(random.randint(150, 255) for _ in range(3))
                              particles.append([sx, sy, vx, vy, 35, color])

                  for sx, sy, stamp, angle, speed in current_shape:
                      for i in range(12):
                          angle = random.uniform(0, 2 * np.pi)
                          speed = random.uniform(2, 7)
                          vx = np.cos(angle) * speed
                          vy = np.sin(angle) * speed
                          color = tuple(random.randint(150, 255) 
                                        for _ in range(3))
                          particles.append([sx, sy, vx, vy, 35, color])

                  drawing = False
                  current_shape = []
                  shapes = []
                  selected_shape = None
                  dragging = False
                  clear_was_active = True

              if gesture != "CLEAR":
                  clear_was_active = False

 
  
  for shape in shapes:
      for i in range(len(shape)):
          sx, sy, stamp, angle, speed = shape[i]

          rotated_stamp = rotate_image(stamp, angle)

          place_image(frame, rotated_stamp, sx, sy)

          shape[i] = (sx, sy, stamp, (angle + speed) % 360, speed)


  for i in range(len(current_shape)):
      sx, sy, stamp, angle, speed = current_shape[i]

      rotated_stamp = rotate_image(stamp, angle)

      place_image(frame, rotated_stamp, sx, sy)

      current_shape[i] = (sx, sy, stamp, (angle + speed) % 360, speed)
  for i in range(len(particles) - 1, -1, -1):
    px, py, vx, vy, life, color = particles[i]

    px += vx
    py += vy
    vy += 0.08
    vx *= 0.98
    life -= 1

    radius = max(1, int(3 * life / 35))

    cv2.circle(frame, (int(px), int(py)), radius, color, -1)

    if life <= 0:
        particles.pop(i)
    else:
        particles[i] = [px, py, vx, vy, life, color]
  cv2.imshow("Camera", frame)

  if cv2.waitKey(1)==ord ('q'):
    break

cap.release()
cv2.destroyAllWindows()
